In [5]:
from docx import Document
from collections import Counter, defaultdict
from lxml import etree


# XML namespaces used in WordprocessingML
NS = {
    "w": "http://schemas.openxmlformats.org/wordprocessingml/2006/main",
    "a": "http://schemas.openxmlformats.org/drawingml/2006/main",
    "r": "http://schemas.openxmlformats.org/officeDocument/2006/relationships",
    "wp": "http://schemas.openxmlformats.org/drawingml/2006/wordprocessingDrawing",
    "c": "http://schemas.openxmlformats.org/drawingml/2006/chart",
}

def _iter_parts(doc: Document):
    """Yield document parts to scan: body + headers + footers."""
    yield ("document", doc.part)

    for si, section in enumerate(doc.sections, start=1):
        if section.header:
            yield (f"header_s{si}", section.header.part)
        if section.footer:
            yield (f"footer_s{si}", section.footer.part)

def _find_referenced_rids_in_part(part):
    """
    Return referenced relationship IDs for:
      - images: a:blip/@r:embed
      - charts: c:chart/@r:id
    """
    root = part._element  # BaseOxmlElement

    img_rids = set()
    chart_rids = set()

    # --- Images ---
    blip_xpath = etree.XPath(".//a:blip", namespaces=NS)
    for blip in blip_xpath(root):
        rid = blip.get(f"{{{NS['r']}}}embed")
        if rid:
            img_rids.add(rid)

    # --- Charts ---
    chart_xpath = etree.XPath(".//c:chart", namespaces=NS)
    for chart in chart_xpath(root):
        rid = chart.get(f"{{{NS['r']}}}id")
        if rid:
            chart_rids.add(rid)

    return img_rids, chart_rids

def referenced_vs_packaged_figures(docx_path):
    doc = Document(docx_path)

    # 1) referenced rIds (what’s actually used)
    referenced_images = set()
    referenced_charts = set()
    where_used = defaultdict(list)

    for part_name, part in _iter_parts(doc):
        img_rids, chart_rids = _find_referenced_rids_in_part(part)
        for rid in img_rids:
            referenced_images.add(rid)
            where_used[rid].append(part_name)
        for rid in chart_rids:
            referenced_charts.add(rid)
            where_used[rid].append(part_name)

    # 2) packaged rIds (what exists in doc.part.rels)
    packaged_images = {rid for rid, rel in doc.part.rels.items() if "image" in rel.reltype}
    packaged_charts = {rid for rid, rel in doc.part.rels.items() if "chart" in rel.reltype}

    # 3) Differences
    unreferenced_images = packaged_images - referenced_images
    unreferenced_charts = packaged_charts - referenced_charts

    # 4) Produce a helpful summary
    summary = {
        "referenced": {
            "images": len(referenced_images),
            "charts": len(referenced_charts),
            "total": len(referenced_images) + len(referenced_charts),
        },
        "packaged_in_main_part_rels": {
            "images": len(packaged_images),
            "charts": len(packaged_charts),
            "total": len(packaged_images) + len(packaged_charts),
        },
        "unreferenced_in_main_part_rels": {
            "images": len(unreferenced_images),
            "charts": len(unreferenced_charts),
            "total": len(unreferenced_images) + len(unreferenced_charts),
        },
        "unreferenced_rids": {
            "images": sorted(unreferenced_images),
            "charts": sorted(unreferenced_charts),
        },
        "where_used_sample": dict(list(where_used.items())[:10]),
    }

    return summary

In [6]:
summary = referenced_vs_packaged_figures(r"\\wsl.localhost\Ubuntu-24.04\home\joe\work\NotBic\Ports\New-Method\docs\BTS_Port-Performance-2026_Annual-Report_DRAFT for BTS_12.5.25_asof_12.10.docx")
print(summary)


{'referenced': {'images': 18, 'charts': 23, 'total': 41}, 'packaged_in_main_part_rels': {'images': 18, 'charts': 23, 'total': 41}, 'unreferenced_in_main_part_rels': {'images': 0, 'charts': 0, 'total': 0}, 'unreferenced_rids': {'images': [], 'charts': []}, 'where_used_sample': {'rId85': ['document'], 'rId11': ['document'], 'rId30': ['document'], 'rId84': ['document'], 'rId15': ['document'], 'rId90': ['document'], 'rId69': ['document'], 'rId35': ['document'], 'rId68': ['document'], 'rId81': ['document']}}
